In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import random
from collections import Counter

# Paths — adjust if your notebook isn't in notebooks/
CAPTIONS_PATH = Path("../data/flickr8k/captions.txt")
IMAGES_DIR = Path("../data/flickr8k/Images")

In [4]:
# src/data.py

from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms
from transformers import DistilBertTokenizer


class Flickr8kDataset(Dataset):
    """
    Returns (image, input_ids, attention_mask) for a given index.
    Each sample is one (image, caption) pair. With ~5 captions per image,
    the same image appears ~5 times in this dataset with different captions.
    """

    def __init__(
        self,
        captions_csv: Path,
        images_dir: Path,
        split_file: Path,
        tokenizer_name: str = "distilbert-base-uncased",
        max_length: int = 32,
        image_size: int = 224,
        is_train: bool = True,
    ):
        # 1. Load all captions, filter to only images in this split
        df = pd.read_csv(captions_csv)
        split_images = set(Path(split_file).read_text().splitlines())
        self.df = df[df['image'].isin(split_images)].reset_index(drop=True)

        self.images_dir = Path(images_dir)
        self.max_length = max_length

        # 2. Tokenizer
        self.tokenizer = DistilBertTokenizer.from_pretrained(tokenizer_name)

        # 3. Image transforms — augment for train, deterministic for eval
        if is_train:
            self.transform = transforms.Compose([
                transforms.Resize(256),
                transforms.RandomCrop(image_size),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225],
                ),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(image_size),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225],
                ),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(self.images_dir / row['image']).convert("RGB")
        image = self.transform(image)

        encoding = self.tokenizer(
            row['caption'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

        return {
            'image': image,
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }

from torch.utils.data import DataLoader


def build_dataloaders(
    captions_csv: Path,
    images_dir: Path,
    splits_dir: Path,
    batch_size: int = 256,
    num_workers: int = 4,
    max_length: int = 32,
    image_size: int = 224,
):
    """Build train/val/test DataLoaders with sensible defaults."""

    train_ds = Flickr8kDataset(
        captions_csv=captions_csv,
        images_dir=images_dir,
        split_file=splits_dir / "train.txt",
        max_length=max_length,
        image_size=image_size,
        is_train=True,
    )
    val_ds = Flickr8kDataset(
        captions_csv=captions_csv,
        images_dir=images_dir,
        split_file=splits_dir / "val.txt",
        max_length=max_length,
        image_size=image_size,
        is_train=False,
    )
    test_ds = Flickr8kDataset(
        captions_csv=captions_csv,
        images_dir=images_dir,
        split_file=splits_dir / "test.txt",
        max_length=max_length,
        image_size=image_size,
        is_train=False,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
        persistent_workers=(num_workers > 0),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False,
        persistent_workers=(num_workers > 0),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False,
        persistent_workers=(num_workers > 0),
    )

    return train_loader, val_loader, test_loader

In [5]:
# In your notebook, or a new cell

import sys
sys.path.append("..")  # so we can import from src/

from pathlib import Path
from src.data import Flickr8kDataset

train_ds = Flickr8kDataset(
    captions_csv=Path("../data/flickr8k/captions_clean.csv"),
    images_dir=Path("../data/flickr8k/Images"),
    split_file=Path("../data/splits/train.txt"),
    is_train=True,
)

print(f"Dataset length: {len(train_ds)}")

sample = train_ds[0]
print(f"Image shape:          {sample['image'].shape}")
print(f"Image dtype:          {sample['image'].dtype}")
print(f"Image min/max:        {sample['image'].min():.3f} / {sample['image'].max():.3f}")
print(f"input_ids shape:      {sample['input_ids'].shape}")
print(f"input_ids dtype:      {sample['input_ids'].dtype}")
print(f"attention_mask shape: {sample['attention_mask'].shape}")
print(f"First 10 input_ids:   {sample['input_ids'][:10].tolist()}")
print(f"First 10 mask values: {sample['attention_mask'][:10].tolist()}")

Dataset length: 29988
Image shape:          torch.Size([3, 224, 224])
Image dtype:          torch.float32
Image min/max:        -1.741 / 2.640
input_ids shape:      torch.Size([32])
input_ids dtype:      torch.int64
attention_mask shape: torch.Size([32])
First 10 input_ids:   [101, 1037, 2304, 3899, 1998, 1037, 7282, 3899, 2024, 3554]
First 10 mask values: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [1]:
# Cell A — path setup
from pathlib import Path
import sys

def find_repo_root():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src").exists() and (parent / "data").exists():
            return parent
    raise RuntimeError("Couldn't find repo root")

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/raydenkhuraijam/projects/mini-clip


In [2]:
# Cell B — DataLoader
import time
from src.data import build_dataloaders

train_loader, val_loader, test_loader = build_dataloaders(
    captions_csv=REPO_ROOT / "data/flickr8k/captions_clean.csv",
    images_dir=REPO_ROOT / "data/flickr8k/Images",
    splits_dir=REPO_ROOT / "data/splits",
    batch_size=32,
    num_workers=2,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

t0 = time.time()
batch = next(iter(train_loader))
t1 = time.time()

print(f"\nFirst batch loaded in {t1-t0:.2f}s")
print(f"  image shape:          {batch['image'].shape}")
print(f"  input_ids shape:      {batch['input_ids'].shape}")
print(f"  attention_mask shape: {batch['attention_mask'].shape}")

Train batches: 937
Val batches:   157
Test batches:  171


/Users/raydenkhuraijam/projects/mini-clip/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



First batch loaded in 2.84s
  image shape:          torch.Size([32, 3, 224, 224])
  input_ids shape:      torch.Size([32, 32])
  attention_mask shape: torch.Size([32, 32])
